# Phase 5 — Evaluation Results

Compare retrieval strategies using RAGAS. Covers:
- dense, sparse, hybrid, hybrid+rerank
- breakdown by question type (factual, aggregation, table, figure)
- failure analysis

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from pathlib import Path

from src.retrieval.vector_store import VectorStore
from src.evaluation.test_dataset import generate_from_chunks, load_dataset
from src.evaluation.ragas_eval import run_full_comparison, evaluate_strategy
from src.retrieval.hybrid_search import RetrievalMode

DATASET_PATH = "../data/processed/qa_dataset.json"
RESULTS_PATH = "../data/processed/eval_results.json"

## 1. Generate QA dataset

Skip if `qa_dataset.json` already exists — generation costs API calls.

In [ ]:
if not Path(DATASET_PATH).exists():
    store = VectorStore()
    pairs = generate_from_chunks(
        store=store,
        output_path=DATASET_PATH,
        questions_per_chunk=2,
        max_chunks=25,
    )
else:
    pairs = load_dataset(DATASET_PATH)
    print(f"Loaded {len(pairs)} QA pairs from cache")

df_pairs = pd.DataFrame([vars(p) for p in pairs])
print(df_pairs['type'].value_counts().to_string())
df_pairs.head()

## 2. Run full comparison

In [ ]:
if not Path(RESULTS_PATH).exists():
    df_results = run_full_comparison(DATASET_PATH, output_path=RESULTS_PATH)
else:
    df_results = pd.read_json(RESULTS_PATH)
    print("Loaded cached results")

df_results

## 3. Results table

In [ ]:
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
available = [m for m in metrics if m in df_results.columns]

display_df = df_results[["mode"] + available].copy()
for col in available:
    display_df[col] = display_df[col].map(lambda x: f"{x:.3f}" if isinstance(x, float) else x)

display_df.style.highlight_max(subset=available, color="#d4edda").set_caption("RAGAS Metrics by Retrieval Mode")

In [ ]:
# Radar chart
if len(available) >= 3:
    angles = np.linspace(0, 2 * np.pi, len(available), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
    colors = cm.tab10.colors

    for i, row in df_results.iterrows():
        vals = [row.get(m, 0) for m in available]
        vals += vals[:1]
        ax.plot(angles, vals, label=row["mode"], color=colors[i % len(colors)])
        ax.fill(angles, vals, alpha=0.1, color=colors[i % len(colors)])

    ax.set_thetagrids(np.degrees(angles[:-1]), available)
    ax.set_ylim(0, 1)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    ax.set_title("RAGAS Metrics — Retrieval Mode Comparison", pad=20)
    plt.tight_layout()
    plt.show()

## 4. Breakdown by question type

In [ ]:
# Run best mode per question type
best_mode = RetrievalMode.HYBRID_RERANK  # update after seeing results

from src.embeddings.text_embedder import TextEmbedder
from src.generation.llm_client import LLMClient
from src.retrieval.hybrid_search import HybridSearcher
from src.retrieval.reranker import Reranker
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

store = VectorStore()
searcher = HybridSearcher(store=store, embedder=TextEmbedder())
reranker = Reranker()
llm = LLMClient()

type_results = []
for qtype in df_pairs['type'].unique():
    subset = [p for p in pairs if p.type == qtype]
    if not subset:
        continue
    questions, answers, contexts, ground_truths = [], [], [], []
    for p in subset:
        chunks = searcher.search(p.question, mode=best_mode, top_k=8, reranker=reranker)
        result = llm.generate(p.question, chunks)
        questions.append(p.question)
        answers.append(result.answer)
        contexts.append([c.content for c in chunks])
        ground_truths.append(p.answer)
    ds = Dataset.from_dict({"question": questions, "answer": answers, "contexts": contexts, "ground_truth": ground_truths})
    scores = evaluate(ds, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
    row = scores.to_pandas().mean().to_dict()
    row["type"] = qtype
    type_results.append(row)

pd.DataFrame(type_results)

## 5. Failure analysis

Low faithfulness = hallucination. Low context_recall = retrieval missing relevant chunks.

In [ ]:
# Re-run best mode and collect per-question scores for inspection
questions, answers, contexts, ground_truths = [], [], [], []

for p in pairs:
    chunks = searcher.search(p.question, mode=best_mode, top_k=8, reranker=reranker)
    result = llm.generate(p.question, chunks)
    questions.append(p.question)
    answers.append(result.answer)
    contexts.append([c.content for c in chunks])
    ground_truths.append(p.answer)

ds_full = Dataset.from_dict({"question": questions, "answer": answers, "contexts": contexts, "ground_truth": ground_truths})
scores_full = evaluate(ds_full, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
df_scores = scores_full.to_pandas()
df_scores["question"] = questions
df_scores["answer"] = answers
df_scores["ground_truth"] = ground_truths

# Worst performers
print("=== Low faithfulness (potential hallucination) ===")
df_scores.nsmallest(5, "faithfulness")[["question", "faithfulness", "answer", "ground_truth"]]

In [ ]:
print("=== Low context_recall (retrieval missing chunks) ===")
df_scores.nsmallest(5, "context_recall")[["question", "context_recall", "ground_truth"]]

## 6. Summary

Fill in after running all cells.

**Best mode:** _TBD_

**Key findings:**
- Table questions: _TBD_ (expected: lower recall since table structure is harder to chunk)
- Figure questions: _TBD_ (expected: depends on caption quality)
- Aggregation questions: _TBD_ (expected: lowest scores — require multi-chunk reasoning)

**Known failure patterns:**
- _Fill after failure analysis_

**What to improve:**
- _Fill after failure analysis_